# 06 - TS-ICL zero-shot imputation

A usage template for applying TS-ICL, a zero-shot, in-context-learning
time-series foundation model, to the chlorophyll reconstruction task. This
notebook is explanatory plus a runnable template -- it requires installing
TS-ICL and its model checkpoint separately, which are not vendored in this
repository.

## License note

TS-ICL code and pretrained weights are distributed under the original
authors' license, separate from this repository's license. Review their
license and terms of use before installing or running TS-ICL.


## 1. Install TS-ICL

```bash
pip install tsicl
```

(Package name and installation instructions are illustrative -- consult the TS-ICL authors' repository for the current installation method and checkpoint download instructions.)

In [ ]:
# Optional: only run this cell if you have installed tsicl and want to execute live.
# import tsicl
# print(tsicl.__version__)


## 2. Checkpoint handling

TS-ICL ships pretrained checkpoints from its authors. Download and point
your model-loading call at the checkpoint path; this repository does not
include or redistribute any TS-ICL weights.

```python
# model = tsicl.load_pretrained(checkpoint_path="/path/to/tsicl_checkpoint")
```


## 3. Prepare the target series with gaps as NaN

In [ ]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, "../src")
from coastal_gap_reconstruction.data_loading import load_daily_target

target_df = load_daily_target("../data_public/chlorophyll/chlorophyll_daily_target.csv")

# Work in log10 space, as used throughout this benchmark
log10_chl = np.log10(target_df["chl_mean"].clip(lower=1e-3))

# Example: artificially hide a 14-day window for a quick illustration
example_start = "2018-03-01"
example_end = "2018-03-14"
series_with_gap = log10_chl.copy()
series_with_gap.loc[example_start:example_end] = np.nan

series_with_gap.loc["2018-02-20":"2018-03-20"]


## 4. Pass covariates through `covars`

The leading TS-ICL configuration in this benchmark uses a satellite
chlorophyll proxy as a single covariate channel. Covariates must be shaped
`(1, T, C)` -- see `src/coastal_gap_reconstruction/tsicl_helpers.py` for why
the explicit batch dimension matters.


In [ ]:
from coastal_gap_reconstruction.data_loading import load_feature_table
from coastal_gap_reconstruction.tsicl_helpers import build_covariate_block

features_df = load_feature_table("../data_public/chlorophyll/chlorophyll_predictor_features_curated.csv")
satellite_proxy = features_df["chl_cons_log10"]

window = satellite_proxy.loc["2018-02-20":"2018-03-20"]
covar_block = build_covariate_block(window.to_numpy()[:, None])
covar_block.shape


## 5. Sparse covariates with `allow_auto_complete=True`

If a covariate channel has its own missing values (common for satellite
products with cloud cover), pass `allow_auto_complete=True` so TS-ICL fills
small covariate gaps internally rather than failing.


In [ ]:
# mean, quantiles = run_tsicl_imputation(
#     model,
#     target_series=series_with_gap.loc["2018-02-20":"2018-03-20"].to_numpy(),
#     covariate_array=window.to_numpy()[:, None],
#     allow_auto_complete=True,
# )


## 6. Quantile outputs and uncertainty bands

TS-ICL returns both a point estimate and a set of quantile predictions
(5th-95th percentile by default), giving an uncertainty band around the
reconstruction. See `results_public/chlorophyll/chlorophyll_reconstruction_tsicl_satellite_proxy.csv`
for an example of the quantile columns (q05...q95) in the precomputed
public output.


## 7. Evaluate only on artificial hidden positions

Score TS-ICL output only against the secretly retained true values of
artificially hidden days (see notebook 02). Never compare against days the
model could see, and never treat real-gap output as validation evidence
(see `docs/evidence_hierarchy.md`).
